# 03 – Exploratory data analysis

Visual exploration of the cleaned dataset. Nothing is fitted here; the failure thresholds drawn in section 5 come from the dataset documentation. All figures are saved to `figures/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from config import CLEAN_DATA, FIG_DIR, TARGET, FAILURE_MODES

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
GRAY, BLUE, ORANGE, RED = "#b9b8b3", "#2a78d6", "#eb6834", "#e34948"

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.edgecolor": MUTED,
    "axes.labelcolor": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "legend.frameon": False,
})


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png")

In [ ]:
df = pd.read_csv(CLEAN_DATA)

sensors = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
short = {
    "Air temperature [K]": "Air temp.",
    "Process temperature [K]": "Process temp.",
    "Rotational speed [rpm]": "Speed",
    "Torque [Nm]": "Torque",
    "Tool wear [min]": "Tool wear",
    TARGET: "Failure",
}

failed = df[TARGET] == 1
speed = df["Rotational speed [rpm]"]
torque = df["Torque [Nm]"]
wear = df["Tool wear [min]"]
temp_diff = df["Process temperature [K]"] - df["Air temperature [K]"]

## 1. Class balance and failure modes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={"width_ratios": [1, 2]})

counts = df[TARGET].value_counts().sort_index()
axes[0].bar(["No failure", "Failure"], counts.values, color=BLUE, width=0.6)
for x, v in enumerate(counts.values):
    axes[0].text(x, v, f"{v:,}\n({v / len(df):.1%})", ha="center", va="bottom", color=INK, fontsize=9)
axes[0].set_ylim(0, counts.max() * 1.25)
axes[0].set_ylabel("Number of jobs")
axes[0].set_title("Class balance")

modes = df[FAILURE_MODES].sum()
axes[1].bar(modes.index, modes.values, color=BLUE, width=0.6)
for x, v in enumerate(modes.values):
    axes[1].text(x, v, str(v), ha="center", va="bottom", color=INK, fontsize=9)
axes[1].set_ylim(0, modes.max() * 1.2)
axes[1].set_ylabel("Number of failures")
axes[1].set_title("Failures per mode (a failure can have several modes)")

for ax in axes:
    ax.grid(axis="x", visible=False)
fig.tight_layout()
save(fig, "03_class_balance")

## 2. Sensor distributions: failure vs no failure

Each histogram is normalised to its own group, so the shapes can be compared despite the class imbalance.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for ax, col in zip(axes.flat, sensors):
    bins = np.histogram_bin_edges(df[col], bins=40)
    ax.hist(df.loc[~failed, col], bins=bins, density=True, color=GRAY, label="No failure")
    ax.hist(df.loc[failed, col], bins=bins, density=True, histtype="step", linewidth=2, color=BLUE, label="Failure")
    ax.set_xlabel(col)
    ax.set_yticks([])
    ax.grid(False)
axes[0, 0].set_ylabel("Density")
axes[1, 0].set_ylabel("Density")
axes.flat[-1].axis("off")
axes.flat[-1].legend(*axes[0, 0].get_legend_handles_labels(), loc="center", fontsize=11)
fig.suptitle("Distribution of each sensor variable, by outcome", y=1.0)
fig.tight_layout()
save(fig, "03_distributions")

## 3. Correlations

In [ ]:
cols = sensors + [TARGET]
corr = df[cols].corr().rename(index=short, columns=short)
cmap = LinearSegmentedColormap.from_list("diverging", [BLUE, "#f0efec", RED])

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.index)
ax.grid(False)
for i in range(len(corr)):
    for j in range(len(corr)):
        value = corr.iloc[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=9,
                color="white" if abs(value) > 0.6 else INK)
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson correlation")
ax.set_title("Correlation matrix")
fig.tight_layout()
save(fig, "03_correlation")

Air and process temperature are strongly correlated, as are speed and torque. Each single variable correlates only weakly with failure, which suggests failures depend on combinations of variables rather than on one variable alone.

## 4. Failure rate by product type

In [ ]:
by_type = df.groupby("Type")[TARGET].agg(["mean", "sum", "size"]).loc[["L", "M", "H"]]

fig, ax = plt.subplots(figsize=(5, 3.4))
ax.bar(by_type.index, by_type["mean"] * 100, color=BLUE, width=0.6)
for x, (rate, n_fail, n) in enumerate(by_type[["mean", "sum", "size"]].itertuples(index=False)):
    ax.text(x, rate * 100, f"{rate:.1%}\n({n_fail}/{n:,})", ha="center", va="bottom", color=INK, fontsize=9)
ax.set_ylim(0, by_type["mean"].max() * 100 * 1.35)
ax.set_xlabel("Product type (L = low, M = medium, H = high quality)")
ax.set_ylabel("Failure rate (%)")
ax.set_title("Failure rate by product type")
ax.grid(axis="x", visible=False)
fig.tight_layout()
save(fig, "03_failure_rate_by_type")
by_type

## 5. Failure mechanisms

Each plot highlights one failure mode against the other failures and the non-failed jobs, with the documented threshold drawn as a dashed line.

In [ ]:
def scatter_mode(ax, x, y, mode):
    is_mode = df[mode] == 1
    other = failed & ~is_mode
    ax.scatter(x[~failed], y[~failed], s=4, color=GRAY, alpha=0.5, label="No failure", rasterized=True)
    ax.scatter(x[other], y[other], s=18, color=ORANGE, edgecolor="white", linewidth=0.5, label="Other failure")
    ax.scatter(x[is_mode], y[is_mode], s=18, color=BLUE, edgecolor="white", linewidth=0.5, label=mode)

### Power failure (PWF): power = torque × angular speed outside 3500–9000 W

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
scatter_mode(ax, speed, torque, "PWF")
rpm = np.linspace(speed.min(), speed.max(), 300)
for power in (3500, 9000):
    ax.plot(rpm, power / (rpm * 2 * np.pi / 60), color=INK, linewidth=1, linestyle="--")
ax.text(rpm[-1], 9000 / (rpm[-1] * 2 * np.pi / 60) + 1.5, "9000 W", color=INK, fontsize=9, ha="right", va="bottom")
ax.text(rpm[0], 3500 / (rpm[0] * 2 * np.pi / 60) - 3.5, "3500 W", color=INK, fontsize=9, ha="left", va="top")
ax.set_ylim(0, torque.max() * 1.05)
ax.set_xlabel("Rotational speed [rpm]")
ax.set_ylabel("Torque [Nm]")
ax.set_title("Power failures occur outside the 3500–9000 W band")
ax.legend(loc="upper right", markerscale=1.5)
fig.tight_layout()
save(fig, "03_mechanism_power")

### Heat dissipation failure (HDF): temperature difference < 8.6 K and speed < 1380 rpm

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
scatter_mode(ax, speed, temp_diff, "HDF")
ax.axhline(8.6, color=INK, linewidth=1, linestyle="--")
ax.axvline(1380, color=INK, linewidth=1, linestyle="--")
ax.text(1.01, 8.6, "8.6 K", color=INK, fontsize=9, ha="left", va="center", transform=ax.get_yaxis_transform())
ax.text(1380, 1.01, "1380 rpm", color=INK, fontsize=9, ha="center", va="bottom", transform=ax.get_xaxis_transform())
ax.set_xlabel("Rotational speed [rpm]")
ax.set_ylabel("Process − air temperature [K]")
ax.set_title("Heat dissipation failures: small temperature margin at low speed", pad=16)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, markerscale=1.5)
fig.tight_layout()
save(fig, "03_mechanism_heat")

### Overstrain failure (OSF): tool wear × torque above a limit that depends on product type

In [ ]:
limits = {"L": 11000, "M": 12000, "H": 13000}
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, (product, limit) in zip(axes, limits.items()):
    rows = df["Type"] == product
    is_mode = df["OSF"] == 1
    other = failed & ~is_mode
    ax.scatter(wear[rows & ~failed], torque[rows & ~failed], s=4, color=GRAY, alpha=0.5, label="No failure", rasterized=True)
    ax.scatter(wear[rows & other], torque[rows & other], s=18, color=ORANGE, edgecolor="white", linewidth=0.5, label="Other failure")
    ax.scatter(wear[rows & is_mode], torque[rows & is_mode], s=18, color=BLUE, edgecolor="white", linewidth=0.5, label="OSF")
    minutes = np.linspace(limit / torque.max(), wear.max(), 200)
    ax.plot(minutes, limit / minutes, color=INK, linewidth=1, linestyle="--")
    ax.set_title(f"Type {product}: limit {limit:,} min·Nm")
    ax.set_xlabel("Tool wear [min]")
axes[0].set_ylabel("Torque [Nm]")
fig.suptitle("Overstrain failures occur above the wear × torque limit")
fig.tight_layout()
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=3, markerscale=1.5, bbox_to_anchor=(0.5, -0.06))
save(fig, "03_mechanism_overstrain")

### Tool wear failure (TWF): only at high tool wear

In [ ]:
bins = np.append(np.arange(0, 240, 10), wear.max() + 1)
wear_bin = pd.cut(wear, bins, right=False)
twf = df.groupby(wear_bin, observed=False)["TWF"].agg(["sum", "size"])
twf_rate = twf["sum"] / twf["size"] * 100

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(bins[:-1], twf_rate.values, width=np.diff(bins) - 1, align="edge", color=BLUE)
for left, width, rate, n_fail, n in zip(bins[:-1], np.diff(bins), twf_rate, twf["sum"], twf["size"]):
    if n_fail > 0:
        ax.text(left + width / 2, rate, f"{n_fail}/{n}", ha="center", va="bottom", color=INK, fontsize=8)
ax.set_ylim(0, twf_rate.max() * 1.2)
ax.set_xlim(160, bins[-1] + 2)
ax.set_xlabel("Tool wear [min] (10-minute bins; last bin 230 min and above; no TWF below 190 min)")
ax.set_ylabel("TWF rate (%)")
ax.set_title("Tool wear failures only occur beyond about 200 minutes")
ax.grid(axis="x", visible=False)
fig.tight_layout()
save(fig, "03_mechanism_tool_wear")

In [ ]:
twf_wear = wear[df["TWF"] == 1]
print(f"TWF failures: {len(twf_wear)}, tool wear {twf_wear.min()}–{twf_wear.max()} min")
print(f"Jobs with tool wear >= {twf_wear.min()} min: {(wear >= twf_wear.min()).sum()}")